In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
import warnings


sys.path.append('../src')
from db_connection import get_connection

warnings.filterwarnings("ignore")

conn = get_connection()
print("Ready!")

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)


In [ ]:
query = """
    SELECT 
        p.product_category_name_english AS category,
        ROUND(SUM(oi.price)::NUMERIC, 2) AS total_revenue,
        COUNT(DISTINCT oi.order_id)      AS total_orders
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    WHERE p.product_category_name_english != 'unknown'
    GROUP BY category
    ORDER BY total_revenue DESC
    LIMIT 15;
"""

df_revenue = pd.read_sql(query, conn)

# Plot
fig, ax = plt.subplots()
sns.barplot(data=df_revenue, x="total_revenue", y="category", palette="viridis", ax=ax)
ax.set_title("Top 15 Categories by Revenue", fontsize=16, fontweight="bold")
ax.set_xlabel("Total Revenue (BRL)")
ax.set_ylabel("Category")
plt.tight_layout()
plt.savefig("../visuals/revenue_by_category.png", dpi=150)
plt.show()

df_revenue

In [ ]:
query = """
    SELECT 
        p.product_category_name_english AS category,
        ROUND(AVG(oi.price)::NUMERIC, 2)         AS avg_price,
        ROUND(AVG(oi.freight_value)::NUMERIC, 2)  AS avg_freight,
        ROUND((AVG(oi.freight_value) / NULLIF(AVG(oi.price), 0) * 100)::NUMERIC, 2) AS freight_pct
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    WHERE p.product_category_name_english != 'unknown'
    GROUP BY category
    ORDER BY freight_pct DESC
    LIMIT 15;
"""

df_freight = pd.read_sql(query, conn)

fig, ax = plt.subplots()
sns.barplot(data=df_freight, x="freight_pct", y="category", palette="rocket", ax=ax)
ax.set_title("Top 15 Categories — Freight as % of Price (Revenue Leak!)", fontsize=16, fontweight="bold")
ax.set_xlabel("Freight Cost as % of Item Price")
ax.set_ylabel("Category")
plt.tight_layout()
plt.savefig("../visuals/freight_vs_price.png", dpi=150)
plt.show()

df_freight

In [ ]:
query = """
    SELECT 
        order_status,
        COUNT(*) AS total,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
    FROM orders
    GROUP BY order_status
    ORDER BY total DESC;
"""

df_status = pd.read_sql(query, conn)

fig, ax = plt.subplots()
ax.pie(
    df_status["total"],
    labels=df_status["order_status"],
    autopct="%1.1f%%",
    startangle=140,
    colors=sns.color_palette("viridis", len(df_status))
)
ax.set_title("Order Status Breakdown", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("../visuals/order_status.png", dpi=150)
plt.show()

df_status

In [ ]:
query = """
    SELECT 
        DATE_TRUNC('month', o.order_purchase_timestamp::TIMESTAMP) AS month,
        ROUND(SUM(oi.price)::NUMERIC, 2) AS monthly_revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY month
    ORDER BY month;
"""

df_trend = pd.read_sql(query, conn)
df_trend["month"] = pd.to_datetime(df_trend["month"])

fig, ax = plt.subplots()
ax.plot(df_trend["month"], df_trend["monthly_revenue"], marker="o", color="#2ecc71", linewidth=2)
ax.fill_between(df_trend["month"], df_trend["monthly_revenue"], alpha=0.2, color="#2ecc71")
ax.set_title("Monthly Revenue Trend (Delivered Orders)", fontsize=16, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue (BRL)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../visuals/monthly_trend.png", dpi=150)
plt.show()

df_trend

In [ ]:
query = """
    SELECT 
        c.customer_unique_id,
        c.customer_city,
        c.customer_state,
        ROUND(SUM(oi.price)::NUMERIC, 2) AS total_spent,
        COUNT(DISTINCT o.order_id)        AS total_orders
    FROM customers c
    JOIN orders o      ON c.customer_id  = o.customer_id
    JOIN order_items oi ON o.order_id   = oi.order_id
    GROUP BY c.customer_unique_id, c.customer_city, c.customer_state
    ORDER BY total_spent DESC
    LIMIT 10;
"""

df_customers = pd.read_sql(query, conn)

fig, ax = plt.subplots()
sns.barplot(data=df_customers, x="total_spent", y="customer_unique_id", palette="Blues_r", ax=ax)
ax.set_title("Top 10 Customers by Revenue", fontsize=16, fontweight="bold")
ax.set_xlabel("Total Spent (BRL)")
ax.set_ylabel("Customer ID")
plt.tight_layout()
plt.savefig("../visuals/top_customers.png", dpi=150)
plt.show()

df_customers

In [ ]:
query = """
    SELECT 
        p.product_id,
        p.product_category_name_english AS category,
        ROUND(SUM(oi.price)::NUMERIC, 2) AS total_revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    WHERE p.product_category_name_english != 'unknown'
    GROUP BY p.product_id, p.product_category_name_english
    ORDER BY total_revenue DESC;
"""

df_pareto = pd.read_sql(query, conn)

# Calculate cumulative revenue percentage
df_pareto["cumulative_revenue"] = df_pareto["total_revenue"].cumsum()
df_pareto["cumulative_pct"]     = df_pareto["cumulative_revenue"] / df_pareto["total_revenue"].sum() * 100
df_pareto["product_pct"]        = range(1, len(df_pareto) + 1)
df_pareto["product_pct"]        = df_pareto["product_pct"] / len(df_pareto) * 100

df_pareto.head(10)

In [ ]:
# Find what % of products generate 80% of revenue
crossover = df_pareto[df_pareto["cumulative_pct"] >= 80].iloc[0]
product_pct_at_80 = crossover["product_pct"]

print(f"📊 80/20 RULE RESULTS:")
print(f"   Top {product_pct_at_80:.1f}% of products generate 80% of revenue")
print(f"   That's {int(len(df_pareto) * product_pct_at_80 / 100)} products out of {len(df_pareto)} total")
print(f"   Total revenue: R${df_pareto['total_revenue'].sum():,.2f}")

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

# Bar chart — individual product revenue
ax1.bar(
    range(len(df_pareto)),
    df_pareto["total_revenue"],
    color="#3498db",
    alpha=0.6,
    label="Product Revenue"
)
ax1.set_xlabel("Products (ranked by revenue)", fontsize=12)
ax1.set_ylabel("Revenue (BRL)", fontsize=12, color="#3498db")
ax1.tick_params(axis='x', which='both', bottom=False, labelbottom=False)

# Line chart — cumulative % on second axis
ax2 = ax1.twinx()
ax2.plot(
    range(len(df_pareto)),
    df_pareto["cumulative_pct"],
    color="#e74c3c",
    linewidth=2.5,
    label="Cumulative Revenue %"
)
ax2.axhline(y=80, color="#e74c3c", linestyle="--", alpha=0.7)
ax2.axvline(x=int(len(df_pareto) * product_pct_at_80 / 100),
            color="gray", linestyle="--", alpha=0.7)
ax2.set_ylabel("Cumulative Revenue %", fontsize=12, color="#e74c3c")
ax2.set_ylim(0, 105)

# Annotation
ax2.annotate(
    f"Top {product_pct_at_80:.1f}% of products\n= 80% of revenue",
    xy=(int(len(df_pareto) * product_pct_at_80 / 100), 80),
    xytext=(int(len(df_pareto) * 0.4), 60),
    fontsize=11,
    color="black",
    arrowprops=dict(arrowstyle="->", color="black")
)

ax1.set_title("Pareto Analysis — 80/20 Rule on Product Revenue", fontsize=16, fontweight="bold")
fig.tight_layout()
plt.savefig("../visuals/pareto_80_20.png", dpi=150)
plt.show()

In [ ]:
# These are products dragging down revenue
bottom_20 = df_pareto.tail(int(len(df_pareto) * 0.2))

print(f"🚨 BOTTOM 20% PRODUCTS:")
print(f"   Count : {len(bottom_20)} products")
print(f"   Total Revenue: R${bottom_20['total_revenue'].sum():,.2f}")
print(f"   Avg Revenue per product: R${bottom_20['total_revenue'].mean():,.2f}")
print(f"\n   Top categories in bottom 20%:")
print(bottom_20["category"].value_counts().head(10))

In [ ]:
#RFM

query = """
    SELECT 
        c.customer_unique_id,
        MAX(o.order_purchase_timestamp::TIMESTAMP)  AS last_purchase,
        COUNT(DISTINCT o.order_id)                   AS frequency,
        ROUND(SUM(oi.price)::NUMERIC, 2)             AS monetary
    FROM customers c
    JOIN orders o       ON c.customer_id  = o.customer_id
    JOIN order_items oi ON o.order_id     = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id;
"""

df_rfm = pd.read_sql(query, conn)

# Recency = days since last purchase
import datetime
snapshot_date = df_rfm["last_purchase"].max() + datetime.timedelta(days=1)
df_rfm["recency"] = (snapshot_date - df_rfm["last_purchase"]).dt.days

df_rfm = df_rfm[["customer_unique_id", "recency", "frequency", "monetary"]]
df_rfm.head()

In [ ]:
# Score 5 = best, 1 = worst
# Recency: lower days = better = score 5
df_rfm["R"] = pd.qcut(df_rfm["recency"],   q=5, labels=[5,4,3,2,1])
df_rfm["F"] = pd.qcut(df_rfm["frequency"].rank(method="first"), q=5, labels=[1,2,3,4,5])
df_rfm["M"] = pd.qcut(df_rfm["monetary"],  q=5, labels=[1,2,3,4,5])

df_rfm["RFM_Score"] = df_rfm["R"].astype(str) + df_rfm["F"].astype(str) + df_rfm["M"].astype(str)
df_rfm["Total_Score"] = df_rfm[["R","F","M"]].astype(int).sum(axis=1)

df_rfm.head(10)

In [ ]:
def assign_segment(row):
    r = int(row["R"])
    f = int(row["F"])
    m = int(row["M"])
    score = r + f + m

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 3 and m >= 3:
        return "Loyal Customers"
    elif r >= 4 and f <= 2:
        return "New Customers"
    elif r <= 2 and f >= 3 and m >= 3:
        return "At Risk"
    elif r == 1 and f == 1:
        return "Lost"
    elif score >= 9:
        return "Potential Loyalists"
    else:
        return "Needs Attention"

df_rfm["Segment"] = df_rfm.apply(assign_segment, axis=1)

# Summary
segment_summary = df_rfm.groupby("Segment").agg(
    customer_count=("customer_unique_id", "count"),
    avg_recency   =("recency",   "mean"),
    avg_frequency =("frequency", "mean"),
    avg_monetary  =("monetary",  "mean"),
    total_revenue =("monetary",  "sum")
).round(2).sort_values("total_revenue", ascending=False)

print(segment_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1 — Customer count per segment
segment_summary["customer_count"].sort_values().plot(
    kind="barh", ax=axes[0], color="#3498db"
)
axes[0].set_title("Customers per Segment", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Number of Customers")

# Chart 2 — Revenue per segment  
segment_summary["total_revenue"].sort_values().plot(
    kind="barh", ax=axes[1], color="#2ecc71"
)
axes[1].set_title("Total Revenue per Segment", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Revenue (BRL)")

plt.suptitle("RFM Customer Segmentation", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("../visuals/rfm_segments.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

colors = {
    "Champions":         "#2ecc71",
    "Loyal Customers":   "#3498db",
    "New Customers":     "#9b59b6",
    "Potential Loyalists":"#f39c12",
    "At Risk":           "#e67e22",
    "Needs Attention":   "#95a5a6",
    "Lost":              "#e74c3c",
}

for segment, group in df_rfm.groupby("Segment"):
    ax.scatter(
        group["recency"],
        group["monetary"],
        label=segment,
        alpha=0.5,
        s=30,
        color=colors.get(segment, "gray")
    )

ax.set_title("RFM Scatter — Recency vs Monetary Value", fontsize=16, fontweight="bold")
ax.set_xlabel("Recency (days since last purchase)")
ax.set_ylabel("Monetary Value (BRL)")
ax.legend(title="Segment", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("../visuals/rfm_scatter.png", dpi=150)
plt.show()

In [ ]:
#conn.close()
#print("Connection closed.")